# 🎬 OpenSource Clipping — Simple Mode

**Hard-coded settings + one input per video.**

1. Run **Cell 1** once to set up the environment and save your fixed config.
2. Run **Cell 2** and paste any YouTube URL to generate clips.

Edit the `SETTINGS` dict in Cell 1 if you want to change defaults (ratio, clips, fonts, provider, etc.).

## ① SETUP — Clone repo, install deps, write fixed config

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 1 — SETUP + FIXED CONFIG
# ═══════════════════════════════════════════════════════════════
import os, subprocess, sys, shutil, json
from pathlib import Path

# ── Google Colab: mount Drive (optional, for model cache) ──
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except Exception:
    pass

# ── Fixed pipeline settings (edit only here) ──
SETTINGS = {
    # API keys
    "GOOGLE_API_KEY": "",   # required if using Gemini fallback
    "HF_TOKEN"      : "",   # optional (split-screen)
    "PEXELS_API_KEY" : "",   # optional (B-roll)
    "NVIDIA_API_KEY" : "",   # optional
    "OLLAMA_API_KEY" : "",   # required for Ollama Cloud

    # Video source defaults
    "SOURCE_PLATFORM" : "youtube",
    "SOURCE_HEIGHT" : "max",

    # Output defaults
    "JUMLAH_CLIP"   : 5,
    "RASIO"         : "9:16",
    "RENDER_HEIGHT" : "1080",

    # AI provider
    "AI_PROVIDER"   : "ollama",      # gemini / nvidia / ollama
    "GEMINI_MODEL"  : "gemini-3-flash-preview",
    "OLLAMA_URL"    : "https://ollama.com",
    "OLLAMA_MODEL"  : "gemini-3-flash-preview:cloud",
    "OLLAMA_FALLBACK_URL"  : "https://ollama.com",
    "OLLAMA_FALLBACK_MODEL" : "gemini-3-flash-preview:cloud",

    # Transcription
    "USE_YT_TRANSCRIPT_API" : False,
    "USE_DLP_SUBS"          : True,
    "WHISPER_MODEL"         : "large-v3",
    "WHISPER_COMPUTE_TYPE"  : "float16",

    # Visual style
    "FONT_STYLE"    : "HORMOZI",
    "FACE_DETECTOR" : "mediapipe",
    "YOLO_SIZE"     : "8m",

    # Content toggles
    "NO_BGM"     : True,
    "NO_BROLL"   : True,
    "NO_SUBS"    : False,
    "NO_KARAOKE" : False,
    "NO_HOOK"    : False,

    # Podcast / advanced
    "SPLIT_SCREEN"   : False,
    "SPLIT_TRIGGER"  : "diarization",
    "CAMERA_SWITCH"  : False,
    "HOOK_V2"        : False,
    "NO_SEGMENT_TRIM": False,
    "SILENCE_TRIM"   : False,

    # Video quality
    "VIDEO_PRESET"     : "ultrafast",
    "VIDEO_SCALE_ALGO" : "bicubic",
    "VIDEO_CQ"         : 23,
    "VIDEO_CRF"        : 20,
    "VIDEO_SHARPEN"    : False,

    # Colab / misc
    "COLAB_CLEANUP"    : True,
    "WORDS_PER_SUB"    : 5,
    "HOOK_DURATION"    : 3,
}

# ── Repo paths ──
REPO_URL = "https://github.com/troublescope/Ashortai.git"
CLONE_DIR = "/content/opensource-clipping"
Path(CLONE_DIR).mkdir(parents=True, exist_ok=True)

# ── Clone / update repo ──
if os.path.isdir(f"{CLONE_DIR}/.git"):
    print("📁 Repo exists — pulling latest…")
    subprocess.run(["git", "-C", CLONE_DIR, "pull", "--ff-only"], check=True)
else:
    print("⬇️  Cloning repo…")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, CLONE_DIR], check=True)

os.chdir(CLONE_DIR)

# ── Install dependencies ──
print("⏳ Installing dependencies…")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

# ── Backward-compat patch: GOOGLE_API_KEY only required for Gemini provider ──
main_py = Path(CLONE_DIR) / "main.py"
main_text = main_py.read_text(encoding="utf-8")
old_check = '    if not cfg.api_key_gemini:'
new_check = '    if cfg.ai_provider == "gemini" and not cfg.api_key_gemini:'
if old_check in main_text and new_check not in main_text:
    main_py.write_text(main_text.replace(old_check, new_check), encoding="utf-8")
    print("🩹 Patched main.py: GOOGLE_API_KEY only required when provider is gemini.")
print("✅ Dependencies installed.")

# ── Write .env file ──
env_lines = [f"{k}={v}" for k, v in SETTINGS.items() if "_API_KEY" in k or k in {"GOOGLE_API_KEY"}]
Path(CLONE_DIR, ".env").write_text("\n".join(env_lines) + "\n", encoding="utf-8")
print("🔐 .env written.")

# ── Restore models from Drive cache if available ──
try:
    DRIVE_CACHE = Path("/content/drive/MyDrive/osc_cache")
    DRIVE_CACHE.mkdir(parents=True, exist_ok=True)
    for m in ["blaze_face_full_range.tflite", "face_yolov8m.pt", "face_yolov8n.pt", "face_yolov8s.pt"]:
        src, dst = DRIVE_CACHE / m, Path(CLONE_DIR) / m
        if src.exists() and not dst.exists():
            shutil.copy2(src, dst)
            print(f"📥 Restored {m}")
except Exception:
    pass

print("\n✅ Setup complete. Run Cell 2 to start clipping.")

## ② RUN — Paste YouTube URL and go

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 2 — RUN PIPELINE (only change URL_VIDEO below)
# ═══════════════════════════════════════════════════════════════
import subprocess, sys, shutil, time
from pathlib import Path

# Paste any YouTube URL here, then run this cell
URL_VIDEO = ""  # ← paste YouTube URL here

if not URL_VIDEO.strip():
    raise ValueError("Please paste a YouTube URL in URL_VIDEO above.")

S = SETTINGS

cmd = [
    sys.executable, "main.py",
    "--url", URL_VIDEO.strip(),
    "--source", S["SOURCE_PLATFORM"],
    "--clips", str(S["JUMLAH_CLIP"]),
    "--ratio", S["RASIO"],
    "--render-height", str(S["RENDER_HEIGHT"]),
    "--source-height", str(S["SOURCE_HEIGHT"]),
    "--font-style", S["FONT_STYLE"],
    "--face-detector", S["FACE_DETECTOR"],
    "--yolo-size", S["YOLO_SIZE"],
    "--ai-provider", S["AI_PROVIDER"],
    "--gemini-model", S["GEMINI_MODEL"],
    "--ollama-url", S["OLLAMA_URL"],
    "--ollama-model", S["OLLAMA_MODEL"],
    "--ollama-fallback-url", S["OLLAMA_FALLBACK_URL"],
    "--ollama-fallback-model", S["OLLAMA_FALLBACK_MODEL"],
    "--whisper-model", S["WHISPER_MODEL"],
    "--whisper-compute-type", S["WHISPER_COMPUTE_TYPE"],
    "--video-preset", S["VIDEO_PRESET"],
    "--video-scale-algo", S["VIDEO_SCALE_ALGO"],
    "--video-cq", str(S["VIDEO_CQ"]),
    "--video-crf", str(S["VIDEO_CRF"]),
    "--words-per-sub", str(S["WORDS_PER_SUB"]),
    "--hook-duration", str(S["HOOK_DURATION"]),
]

def add(flag, key):
    if S.get(key):
        cmd.append(flag)

add("--colab-cleanup", "COLAB_CLEANUP")
add("--use-yt-transcript-api", "USE_YT_TRANSCRIPT_API")
add("--use-dlp-subs", "USE_DLP_SUBS")
add("--no-bgm", "NO_BGM")
add("--no-broll", "NO_BROLL")
add("--no-subs", "NO_SUBS")
add("--no-karaoke", "NO_KARAOKE")
add("--no-hook", "NO_HOOK")
add("--camera-switch", "CAMERA_SWITCH")
add("--hook-v2", "HOOK_V2")
add("--no-segment-trim", "NO_SEGMENT_TRIM")
add("--silence-trim", "SILENCE_TRIM")
add("--video-sharpen", "VIDEO_SHARPEN")

if S.get("SPLIT_SCREEN"):
    cmd.append("--split-screen")
    cmd += ["--split-trigger", S["SPLIT_TRIGGER"]]

print("🚀 Starting pipeline…")
print("URL:", URL_VIDEO.strip())
print("Command:", " ".join(cmd))
print("─" * 60)

start = time.time()
result = subprocess.run(cmd, cwd=CLONE_DIR)
if result.returncode != 0:
    print(f"❌ Pipeline exited with code {result.returncode}")
elapsed = time.time() - start

print("─" * 60)
print(f"⏱️ Total time: {elapsed/60:.1f} minutes")

out_dir = Path(CLONE_DIR) / "outputs"
if out_dir.exists():
    files = sorted(out_dir.glob("*_ready.mp4")) + sorted(out_dir.glob("*.jpg"))
    if files:
        print(f"\n📦 Output files ({len(files)}):")
        for f in files:
            print(f"   • {f.name} ({f.stat().st_size/1024/1024:.1f} MB)")
    if list(out_dir.iterdir()):
        zip_path = Path(CLONE_DIR) / "outputs.zip"
        shutil.make_archive(str(zip_path.with_suffix("")), "zip", root_dir=str(out_dir))
        print(f"\n🗜️ outputs.zip → {zip_path.stat().st_size/1024/1024:.1f} MB")
        try:
            from google.colab import files
            files.download(str(zip_path))
        except Exception:
            pass
else:
    print("⚠️ outputs/ directory not found.")